# Gemini Embedding 2 Probe

Small manual notebook for checking the real `gemini-embedding-2` API before wiring it into the library pipeline.

This notebook reads `GOOGLE_API_KEY` or `GEMINI_API_KEY` from the repo-root `.env` file. It never prints the key.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
from google import genai
from google.genai import types

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 768
ROOT

PosixPath('/Users/eltonli/code/rs-demo')

In [2]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


load_env_file(ROOT / ".env")
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
assert api_key, "Set GOOGLE_API_KEY or GEMINI_API_KEY in .env first."

client = genai.Client(api_key=api_key)
print("API key loaded:", bool(api_key))
print("Model:", MODEL)

API key loaded: True
Model: gemini-embedding-2


## Text Embedding

For `gemini-embedding-2`, Google recommends putting the retrieval task in the text itself rather than using the old `task_type` field.

In [3]:
text_query = "task: search result | query: I want a smoky Islay whisky with peat and sea salt."

text_result = client.models.embed_content(
    model=MODEL,
    contents=text_query,
    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
)

text_vector = np.array(text_result.embeddings[0].values, dtype=np.float32)
print("shape:", text_vector.shape)
print("norm:", float(np.linalg.norm(text_vector)))
print("first 8 values:", np.round(text_vector[:8], 6).tolist())

shape: (768,)
norm: 1.0000001192092896
first 8 values: [-0.011617000214755535, 0.000375000003259629, 0.002004999900236726, -0.028589999303221703, -0.024994999170303345, -0.036428000777959824, -0.013260999694466591, 0.004257999826222658]


## Product Text Similarity Smoke Test

This embeds one known catalogue product as a document and compares it with the query above. This is not evaluation yet; it only checks the API/vector mechanics.

In [4]:
from rs_demo.embeddings import build_product_text, load_product_records

records = load_product_records(ROOT / "data/extracted/product_parse_sample.jsonl")
ardbeg = next(record for record in records if record["product_id"] == "p0021-ardbeg-10-year-old")
product_text = "title: ARDBEG 10-YEAR-OLD | text: " + build_product_text(ardbeg)

product_result = client.models.embed_content(
    model=MODEL,
    contents=product_text,
    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
)

product_vector = np.array(product_result.embeddings[0].values, dtype=np.float32)
similarity = float(text_vector @ product_vector / (np.linalg.norm(text_vector) * np.linalg.norm(product_vector)))
print("product_id:", ardbeg["product_id"])
print("cosine similarity:", round(similarity, 6))

product_id: p0021-ardbeg-10-year-old
cosine similarity: 0.743684


## Image And Image+Text Embeddings

This uses one manually cropped movie-scene bottle image and then the full scene plus a natural question.

In [5]:
crop_path = ROOT / "data/eval/movie_scene_bottle_crops/msq008-constantine-ardbeg.crop.jpg"
scene_path = ROOT / "data/eval/movie_scene_query_images/msq008-constantine-ardbeg.jpg"
assert crop_path.exists(), crop_path
assert scene_path.exists(), scene_path

crop_result = client.models.embed_content(
    model=MODEL,
    contents=[types.Part.from_bytes(data=crop_path.read_bytes(), mime_type="image/jpeg")],
    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
)

scene_question_result = client.models.embed_content(
    model=MODEL,
    contents=[
        "What is the whisky in this image?",
        types.Part.from_bytes(data=scene_path.read_bytes(), mime_type="image/jpeg"),
    ],
    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
)

crop_vector = np.array(crop_result.embeddings[0].values, dtype=np.float32)
scene_question_vector = np.array(scene_question_result.embeddings[0].values, dtype=np.float32)

print("crop shape:", crop_vector.shape, "norm:", float(np.linalg.norm(crop_vector)))
print("scene+question shape:", scene_question_vector.shape, "norm:", float(np.linalg.norm(scene_question_vector)))

crop shape: (768,) norm: 0.9999997019767761
scene+question shape: (768,) norm: 0.9999997019767761


## What To Check

- All vectors should have shape `(768,)`.
- Norms should be near `1.0` for `gemini-embedding-2` with truncated dimensions.
- The image and image+text cells are the important multimodal smoke test for our benchmark.